# Check postprocessing

Run the cells in order: prepare model IDs, separate merged ventricles,
fill brain holes, remove small brain objects, then restore original ROI IDs.
The final cell applies the same operations through `FullVolumePostprocessor`.

The default synthetic prediction has a known lesion side, two ventricles,
a third ventricle, a brain hole and a small island. It requires no checkpoint
and makes cleanup visible. Use `INPUT_MODE="normalized"` for a random real
scan (annotations are only a **demonstration input**, not model predictions),
or `"files"` for MRI and saved ROI/brain masks. Choose `TASK="mouse_full"` for
eleven-class outputs; its predicted sides are preserved, without ventricular splitting.

All inputs first use the shared RIA reference. Real inputs also use the
training units/resampling convention; cleanup thresholds are voxel counts on
that 0.1 mm grid. ROI masks are not treated as binary brain masks. This
notebook does not compute performance metrics or modify source files.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT: Path = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "src/preprocessing/preprocessor.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import nibabel as nib
import numpy as np
from src.utils.inspection import InspectionVolume, sample_normalized_scan, show_volumes
from src.preprocessing.preprocessor import Reorient

REFERENCE: Path = (
    REPO_ROOT / "example" / "RARE" / "TBI_fm_19_49"
    / "Anat" / "TBI_fm_19_49_N4.nii.gz"
)
ref_img = nib.load(REFERENCE)
print("Reference axes:", nib.aff2axcodes(ref_img.affine))

from typing import Literal
from src.utils.inspection import postprocessing_example
from src.experiments.config import LabelConfig, PostprocessingConfig
from src.preprocessing.preprocessor import CorrectX10, MapLabels, Resample
from src.evaluation.postprocessing import (
    FullVolumePostprocessor, OutputLabels,
    ipsi_contra_division_callback, morphology_refinement_callback,
)

INPUT_MODE: Literal["example", "normalized", "files"] = "example"
TASK: Literal["tbi", "mouse_full"] = "tbi"
INPUT_LABEL_SPACE: Literal["original", "model"] = "original"
DATA_ROOT: Path = Path("D:/Other files/TESI_CNN/MANUSCRIPT_DATASET/normalized")
DATASET: str = "D1"
SEED: int = 42
IMAGE_PATH: Path | None = None
ROI_PATH: Path | None = None
BRAIN_MASK_PATH: Path | None = None
SPACING_MM: tuple[float, float, float] = (0.1, 0.1, 0.1)
VIEW_AXIS: int = 2
VIEW_POSITION: float = 0.5
USE_CENTROIDS: bool = False
DEFAULT_HEMISPHERE: Literal["right", "left"] = "right"
LESION_VOXELS_THRESHOLD: int = 10
HOLES_MAX_AREA: int = 20000
OBJECT_MIN_AREA: int = 30000

def show_pair(before: InspectionVolume, after: InspectionVolume, title: str) -> None:
    """Compare one postprocessing operation with the same view settings."""
    views = {"Before": before, title: after}
    show_volumes(views, axis=VIEW_AXIS, position=VIEW_POSITION)


## Load and reorient the input

In [ ]:
if INPUT_MODE == "example":
    raw = postprocessing_example()
elif INPUT_MODE == "normalized":
    scan = sample_normalized_scan(root=DATA_ROOT, dataset=DATASET, seed=SEED)
    raw = InspectionVolume(scan.image, scan.labels, scan.brain_mask)
    print("Scan:", scan.id, "| original ID:", scan.original_id)
elif INPUT_MODE == "files":
    if IMAGE_PATH is None or ROI_PATH is None or BRAIN_MASK_PATH is None:
        raise ValueError("Set IMAGE_PATH, ROI_PATH and BRAIN_MASK_PATH")
    image = nib.load(IMAGE_PATH)
    roi = nib.load(ROI_PATH)
    brain = nib.load(BRAIN_MASK_PATH)
    raw = InspectionVolume(image, roi, brain)
else:
    raise ValueError("INPUT_MODE must be example, normalized or files")

reorientation = Reorient(ref_img=ref_img)
source = raw.preprocess([reorientation], seed=SEED)
if INPUT_MODE != "example":
    unit_correction = CorrectX10(reference_resolution=SPACING_MM)
    resampling = Resample(target_resolution=SPACING_MM, interpolation=1, orthogonal=True)
    source = source.preprocess([unit_correction, resampling], seed=SEED)
if source.brain_mask is None:
    raise ValueError("Choose an input with a brain mask")
show_volumes({"Input": source}, axis=VIEW_AXIS, position=VIEW_POSITION)


## Prepare model class IDs

Original-ID inputs are mapped to the chosen model vocabulary. Set
`INPUT_LABEL_SPACE="model"` only for files that already contain raw model IDs.
Step displays explicitly show intermediate class IDs; final results always
restore original annotation IDs. Extra source anatomy maps to task background.

In [ ]:
if TASK == "tbi":
    labels = LabelConfig()
elif TASK == "mouse_full":
    mousefull_mapping: dict[int, int] = {
        0: 0, 1: 1, 2: 2, 3: 3, 5: 4, 6: 5,
        12: 6, 13: 7, 15: 8, 16: 9, 21: 10,
    }
    mousefull_names: tuple[str, ...] = (
        "Background", "Lesion", "Contra CC", "Contra ventricle", "Contra hippocampus",
        "Contra cortex", "Ipsi CC", "Ipsi ventricle", "Ipsi hippocampus", "Ipsi cortex", "Third ventricle",
    )
    labels = LabelConfig(mapping=mousefull_mapping, names=mousefull_names)
else:
    raise ValueError("TASK must be tbi or mouse_full")

model_input = source
if INPUT_LABEL_SPACE == "original":
    mapping = MapLabels(mapping=labels.mapping, unknown="background")
    model_input = source.preprocess([mapping], seed=SEED)
elif INPUT_LABEL_SPACE != "model":
    raise ValueError("INPUT_LABEL_SPACE must be original or model")
model_regions = model_input.labels.get_fdata().astype(np.int16)
model_brain = model_input.brain_mask.get_fdata().astype(np.int16)
allowed_ids = list(range(len(labels.names)))
if not np.isin(model_regions, allowed_ids).all():
    raise ValueError("Input ROI IDs do not match the chosen task")
show_volumes({"Model class IDs": model_input}, axis=VIEW_AXIS, position=VIEW_POSITION)


## Separate merged ventricles (TBI only)

The historical splitter uses the predicted third ventricle to fit a symmetry
plane and the largest valid lesion to identify its side. Output classes are
2=contra ventricle, 3=ipsi ventricle, 4=third ventricle. It needs at least three
third-ventricle voxels when lateral ventricles are present. Without a valid lesion,
historical `DEFAULT_HEMISPHERE="right"` selects the side containing array origin;
this is not an affine-derived anatomical right-side definition.

In [ ]:
split_ventricles: bool = TASK == "tbi"
separated_regions = model_regions.copy()
if split_ventricles:
    split = ipsi_contra_division_callback(
        use_centroids=USE_CENTROIDS, default_hemisphere=DEFAULT_HEMISPHERE,
        lesion_voxels_threshold=LESION_VOXELS_THRESHOLD,
    )
    separated_regions = split(model_regions)
else:
    print("MouseFull keeps its directly predicted ventricle sides.")
separated = model_input.with_masks(separated_regions)
show_pair(model_input, separated, "Separated ventricle class IDs")


## Fill small holes in the binary brain mask

In [ ]:
fill_holes = morphology_refinement_callback(
    fill_small_holes=True, remove_small_objects=False, holes_max_area=HOLES_MAX_AREA,
)
filled_brain = fill_holes(model_brain)
hole_filled = separated.with_masks(separated_regions, filled_brain)
show_pair(separated, hole_filled, "Brain holes filled")


## Remove small objects from the binary brain mask

In [ ]:
remove_objects = morphology_refinement_callback(
    fill_small_holes=False, remove_small_objects=True, object_min_area=OBJECT_MIN_AREA,
)
cleaned_brain = remove_objects(filled_brain)
cleaned = separated.with_masks(separated_regions, cleaned_brain)
show_pair(hole_filled, cleaned, "Small brain objects removed")


## Restore original annotation IDs

The TBI output becomes 0, 1, 3, 13, 21. MouseFull restores its eleven original
values. The brain stays binary 0/1; its cleanup never overwrites categorical ROIs.

In [ ]:
output_labels = OutputLabels.resolve(
    labels, split_ventricles=split_ventricles, original_label_ids=True,
)
restore_ids = MapLabels(mapping=output_labels.prediction_mapping)
restored = cleaned.preprocess([restore_ids], seed=SEED)
print("Original IDs:", output_labels.names)
show_pair(cleaned, restored, "Original annotation IDs")


## Custom combined postprocessing

Edit the settings below and rerun this cell. It starts from raw model IDs,
rather than applying postprocessing twice to an already processed result.
The helper is the same one used in full-volume training/checkpoint evaluation.

In [ ]:
settings = PostprocessingConfig(
    clean_brain_mask=True,
    fill_small_holes=True, holes_max_area=HOLES_MAX_AREA,
    remove_small_objects=True, object_min_area=OBJECT_MIN_AREA,
    split_ventricles=split_ventricles, use_centroids=USE_CENTROIDS,
    default_hemisphere=DEFAULT_HEMISPHERE,
    lesion_voxels_threshold=LESION_VOXELS_THRESHOLD,
)
postprocessor = FullVolumePostprocessor(labels, settings, original_label_ids=True)
processed = postprocessor.process(model_regions, model_brain)
combined = model_input.with_masks(processed.regions, processed.brain_mask)
show_pair(model_input, combined, "Combined result / original annotation IDs")
